In [1]:
import torchaudio

bundle = torchaudio.pipelines.HUBERT_BASE

model = bundle.get_model()


In [2]:
model

Wav2Vec2Model(
  (feature_extractor): FeatureExtractor(
    (conv_layers): ModuleList(
      (0): ConvLayerBlock(
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
      )
      (1-4): 4 x ConvLayerBlock(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
      )
      (5-6): 2 x ConvLayerBlock(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
      )
    )
  )
  (encoder): Encoder(
    (feature_projection): FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (pos_conv_embed): ConvolutionalPositionalEmbedding(
        (conv): ParametrizedConv1d(
          768, 768, kernel_size=(128,), stride=(1,), padding=(64,), groups=16
          (parametriza

In [3]:
audio_path = "../../media/audio.wav"

waveform, sr = torchaudio.load(audio_path)

In [4]:
waveform.shape

torch.Size([1, 161280])

In [5]:
sr

24000

In [6]:
audio_len_seconds = waveform.shape[1] / sr
audio_len_seconds

6.72

In [ ]:
waveform = torchaudio.functional.resample(waveform, sr, bundle.sample_rate)
features, _ = model.extract_features(waveform)

In [8]:
for feat in features:
    print(feat.shape)

torch.Size([1, 335, 768])
torch.Size([1, 335, 768])
torch.Size([1, 335, 768])
torch.Size([1, 335, 768])
torch.Size([1, 335, 768])
torch.Size([1, 335, 768])
torch.Size([1, 335, 768])
torch.Size([1, 335, 768])
torch.Size([1, 335, 768])
torch.Size([1, 335, 768])
torch.Size([1, 335, 768])
torch.Size([1, 335, 768])


In [9]:
len(features)

12

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class HubertEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        bundle = torchaudio.pipelines.HUBERT_BASE
        self.backbone = bundle.get_model()
        self.sample_rate = bundle.sample_rate

    def forward(self, waveform):
        # Expects 16kHz
        features, _ = self.backbone.extract_features(waveform)
        acoustic_features = torch.stack(features[0:3]).mean(dim=0) # Layers 0, 1, 2
        semantic_features = torch.stack(features[9:12]).mean(dim=0) # Layers 9, 10, 11
        return acoustic_features, semantic_features

wav_lm_encoder = HubertEncoder()
audio_path = "../../media/audio.wav"
waveform, sr = torchaudio.load(audio_path)
waveform = torchaudio.functional.resample(waveform, sr, bundle.sample_rate).squeeze()
waveform_batch = torch.stack([waveform, waveform])
wav_lm_encoder(waveform_batch)[0].shape

torch.Size([2, 335, 768])

In [11]:
class FusionDownProj(nn.Module):
    def __init__(self, n_levels):
        super().__init__()
        self.down_proj = nn.Linear(768 * 2, n_levels)
        self.n_levels = n_levels
    
    def forward(self, acoustic, semantic):
        # acoustic: B, T, dim
        # semantic: B, T, dim
        x = torch.cat([acoustic, semantic], dim=2) # B, T, dim * 2
        x = self.down_proj(x) # B * T, n_levels
        x = F.gelu(x)
        return x

acoustic, semantic = wav_lm_encoder(waveform_batch)
fusing_down_proj = FusionDownProj(n_levels=5)
x = fusing_down_proj(acoustic, semantic)
x.shape

torch.Size([2, 335, 5])

In [12]:
class Quantizer(nn.Module):
    def __init__(self, n_levels):
        super().__init__()
        self.n_levels = n_levels
        self.register_buffer('levels', torch.linspace(-1, 1, n_levels))

    def forward(self, x):
        # x: [B, 1]
        x = torch.tanh(x)
        distances = torch.abs(self.levels - x)
        best_matching_index = distances.sort().indices[:, 0]
        x_quantized = self.levels[best_matching_index].unsqueeze(1)
        x_quantized_st = x + (x_quantized - x).detach() # gradient = 1 hack
        return x_quantized_st, best_matching_index.unsqueeze(1) # [B, 1]

In [ ]:
class FSQ(nn.Module):
    def __init__(self, levels: list[int]):
        super().__init__()
        multiplier = []
        for index in range(len(levels)):
            sub_levels = levels[:index]
            if sub_levels is None:
                multiplier.append(torch.tensor([0], dtype=torch.long))
            multiplier.append(torch.prod(torch.tensor(sub_levels, dtype=torch.long)))
        self.register_buffer('multiplier', torch.tensor(multiplier))

        self.quantizers = nn.ModuleList()
        for level in levels:
            self.quantizers.append(Quantizer(level))

    def forward(self, x):
        # x: B, T, n_levels
        x_flatten = torch.flatten(x, start_dim=0, end_dim=1) # B * T, n_levels

        y = []
        z = []
        for i, quantizer in enumerate(self.quantizers):
            x_quantized_st, code = quantizer(x_flatten[:, i].unsqueeze(1))
            y.append(code)
            z.append(x_quantized_st)

        y = torch.stack(y).squeeze() # n_levels, B * T
        y = y.permute(1, 0) # B * T, n_levels
        y = y * self.multiplier # B * T, n_levels
        code = torch.sum(y, dim=1) # B * T
        z = torch.stack(z).squeeze()
        return z.reshape(x.shape), code.reshape(x.shape[0], x.shape[1])


acoustic, semantic = wav_lm_encoder(waveform_batch)
fusing_down_proj = FusionDownProj(n_levels=8)
x = fusing_down_proj(acoustic, semantic)
fsq = FSQ(levels=[4, 4, 4, 4, 4, 4, 4, 4])
fsq(x)[0]

tensor([[[ 0.1429,  0.1429,  0.1429,  ...,  0.1429,  0.1429,  0.1429],
         [ 0.1429,  0.1429,  0.1429,  ...,  0.1429, -0.1429, -0.1429],
         [ 0.1429,  0.1429,  0.1429,  ..., -0.1429,  0.1429,  0.1429],
         ...,
         [-0.1429, -0.1429,  0.1429,  ..., -0.1429, -0.1429, -0.1429],
         [-0.1429, -0.1429, -0.1429,  ..., -0.1429,  0.1429,  0.1429],
         [ 0.1429,  0.1429,  0.1429,  ...,  0.1429,  0.1429,  0.1429]],

        [[ 0.3333, -0.3333, -0.3333,  ...,  0.3333,  0.3333,  0.3333],
         [ 0.3333, -0.3333, -0.3333,  ..., -0.3333, -0.3333,  0.3333],
         [ 0.3333,  0.3333,  0.3333,  ..., -0.3333, -0.3333,  0.3333],
         ...,
         [ 0.3333, -0.3333, -0.3333,  ..., -0.3333, -0.3333, -0.3333],
         [ 0.3333,  0.3333,  0.3333,  ...,  0.3333,  0.3333,  0.3333],
         [ 0.3333,  0.3333,  0.3333,  ..., -0.3333, -0.3333, -0.3333]]],
       grad_fn=<ViewBackward0>)

In [26]:
class DecoderTransformer(nn.Module):
    def __init__(self, input_dim, model_dim=256, num_heads=8, num_layers=4, output_dim=100):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, model_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=model_dim * 4,
            dropout=0.1,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(model_dim, output_dim)
        self.vocos_tokens_per_second = 94
        self.hubert_tokens_per_second = 50

    def forward(self, x):
        # x: B, T_tokens, D
        x = self.input_proj(x)
        x = self.transformer(x)
        x = self.output_proj(x)
        x = x.transpose(1, 2)  # B, n_mels, T_tokens
        audio_length = x.shape[2] / self.hubert_tokens_per_second
        target_frames = audio_length * self.vocos_tokens_per_second
        x = F.interpolate(x, size=int(target_frames), mode='linear', align_corners=False)
        return x


acoustic, semantic = wav_lm_encoder(waveform_batch)
fusing_down_proj = FusionDownProj(n_levels=8)
x = fusing_down_proj(acoustic, semantic)
fsq = FSQ(levels=[8, 8, 8, 8, 4, 4, 4, 4])
x = fsq(x)[0]
print(x.shape)
transformer = DecoderTransformer(8)
x = transformer(x)
print(x.shape)

torch.Size([2, 335, 8])
torch.Size([2, 100, 629])


/tmp/ipykernel_18597/581282021.py:14: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


In [ ]:
import math

from vocos import Vocos

VOCOS_SAMPLE_RATE = 24000
VOCOS_HOP_LENGTH = 256
VOCOS_MEL_BINS = 100

vocos = Vocos.from_pretrained("charactr/vocos-mel-24khz")

acoustic, semantic = wav_lm_encoder(waveform_batch)
fusing_down_proj = FusionDownProj(n_levels=8)
x = fusing_down_proj(acoustic, semantic)
fsq = FSQ(levels=[8, 8, 8, 8, 4, 4, 4, 4])
x = fsq(x)[0]
transformer = DecoderTransformer(8)
x = transformer(x)
audio = vocos.decode(x)

torch.Size([2, 100, 629])


/tmp/ipykernel_18597/581282021.py:14: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


In [30]:
audio.shape

torch.Size([2, 160768])